In [ ]:

import copy, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

In [ ]:
SEED = 42
D_MODEL, N_HEADS, N_LAYERS, DROPOUT = 64, 4, 6, 0.2   # your script's defaults except d_model (notes say 64)
BASE_LR, BASE_WD, BASE_EPOCHS, BASE_PATIENCE = 1e-3, 1e-3, 50, 4
FT_LR,   FT_WD,   FT_EPOCHS,   FT_PATIENCE   = 1e-6, 1e-4, 200, 20   # FT_EPOCHS is a cap to save time, state it in the report
THRESHOLD = 0.5
FOLDS_TO_RUN = [2,3,4,5]        # e.g. [1] to time one fold first; None = all 5

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cuda


In [ ]:
# ---------- data ----------
data = np.load('data.npz')
X_train, y_train = data['X_train'], data['y_train']
NUM_FEATURES = X_train.shape[1]
print("X_train:", X_train.shape, "| positive rate:", round(float(y_train.mean()), 3))  # should be ~0.5 if balanced

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def make_loader(X, y, batch_size, shuffle):
    return DataLoader(TabularDataset(X, y), batch_size=batch_size, shuffle=shuffle)

X_train.shape, y_train.shape

X_train: (45243, 28) | positive rate: 0.5


((45243, 28), (45243,))

In [ ]:

# ---------- model (same as your script) ----------
class FTTransformer(nn.Module):
    def __init__(self, num_features, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.num_features = num_features
        self.d_model = d_model
        # per-feature embedding: one weight + bias vector per feature
        self.feature_weight = nn.Parameter(torch.randn(num_features, d_model) * 0.02)
        self.feature_bias = nn.Parameter(torch.zeros(num_features, d_model))
        # learnable [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        # self-attention + feed-forward (GELU) blocks
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, activation='gelu', batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1))

    def forward(self, x):
        batch_size = x.size(0)
        tokens = x.unsqueeze(-1) * self.feature_weight + self.feature_bias
        cls = self.cls_token.expand(batch_size, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        encoded = self.encoder(tokens)
        cls_out = self.norm(encoded[:, 0, :])
        return self.head(cls_out).squeeze(-1)

In [ ]:

# ---------- training helpers ----------
def evaluate_loss(model, loader, criterion):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            total += criterion(model(xb), yb).item() * xb.size(0)
    return total / len(loader.dataset)

def train_stage(model, train_loader, es_loader, lr, weight_decay,
                max_epochs, patience, use_scheduler):
    """One training stage; early-stops on es_loader and restores the best weights."""
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = (torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)
                 if use_scheduler else None)
    best_loss, best_state, wait = float('inf'), copy.deepcopy(model.state_dict()), 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        es_loss = evaluate_loss(model, es_loader, criterion)
        if scheduler: scheduler.step(es_loss)
        if es_loss < best_loss:
            best_loss, best_state, wait = es_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    print(f"   stage finished after {epoch+1} epochs | best early-stop loss {best_loss:.4f}")
    return model

def predict_proba(model, X):
    model.eval()
    loader = make_loader(X, np.zeros(len(X)), 512, False)
    probs = []
    with torch.no_grad():
        for xb, _ in loader:
            probs.extend(torch.sigmoid(model(xb.to(device))).cpu().numpy())
    return np.array(probs)

In [ ]:

# ---------- 5-fold CV ----------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = [[0.7465,0.7167,0.8152,0.7628,0.8220]]

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train), 1):
    if FOLDS_TO_RUN and fold not in FOLDS_TO_RUN:
        continue
    t0 = time.time()
    print(f"\n===== Fold {fold} =====")
    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_va, y_va = X_train[va_idx], y_train[va_idx]

    # small split inside the training part, used ONLY for early stopping
    X_fit, X_es, y_fit, y_es = train_test_split(
        X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=SEED)
    fit_loader = make_loader(X_fit, y_fit, 256, True)
    es_loader = make_loader(X_es, y_es, 512, False)

    # fresh random weights every fold
    torch.manual_seed(SEED)
    model = FTTransformer(NUM_FEATURES, D_MODEL, N_HEADS, N_LAYERS, DROPOUT).to(device)

    # stage 1: base training (default lr / weight decay)
    model = train_stage(model, fit_loader, es_loader, BASE_LR, BASE_WD,
                        BASE_EPOCHS, BASE_PATIENCE, use_scheduler=True)
    # stage 2: fine-tune from this fold's own base model
    model = train_stage(model, fit_loader, es_loader, FT_LR, FT_WD,
                        FT_EPOCHS, FT_PATIENCE, use_scheduler=False)

    # score on the held-out fold only (never used for training or early stopping)
    probs = predict_proba(model, X_va)
    preds = (probs >= THRESHOLD).astype(int)
    row = [accuracy_score(y_va, preds), precision_score(y_va, preds),
           recall_score(y_va, preds), f1_score(y_va, preds), roc_auc_score(y_va, probs)]
    results.append(row)
    print(f"Fold {fold}: acc {row[0]:.4f} | prec {row[1]:.4f} | rec {row[2]:.4f} | "
          f"f1 {row[3]:.4f} | auc {row[4]:.4f}  ({(time.time()-t0)/60:.1f} min)")

results = np.array(results)
print("\n================ OVERALL 5-FOLD CV RESULTS ================")
for name, m, s in zip(["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
                      results.mean(axis=0), results.std(axis=0)):
    print(f"{name+':':<10} {m:.4f} (+/- {s:.4f})")


===== Fold 2 =====
   stage finished after 6 epochs | best early-stop loss 0.5153
   stage finished after 25 epochs | best early-stop loss 0.5143
Fold 2: acc 0.7443 | prec 0.7112 | rec 0.8228 | f1 0.7629 | auc 0.8232  (1.5 min)

===== Fold 3 =====
   stage finished after 25 epochs | best early-stop loss 0.5001
   stage finished after 23 epochs | best early-stop loss 0.4997
Fold 3: acc 0.7510 | prec 0.7392 | rec 0.7759 | f1 0.7571 | auc 0.8298  (2.3 min)

===== Fold 4 =====
   stage finished after 22 epochs | best early-stop loss 0.4944
   stage finished after 24 epochs | best early-stop loss 0.4943
Fold 4: acc 0.7509 | prec 0.7295 | rec 0.7975 | f1 0.7620 | auc 0.8289  (2.2 min)

===== Fold 5 =====
   stage finished after 10 epochs | best early-stop loss 0.5129
   stage finished after 73 epochs | best early-stop loss 0.5119
Fold 5: acc 0.7512 | prec 0.7227 | rec 0.8152 | f1 0.7662 | auc 0.8250  (3.9 min)

================ OVERALL 5-FOLD CV RESULTS ================
Accuracy:  0.7488 (+